<a href="https://colab.research.google.com/github/kinshukpcs24-ux/AI_Lab_Programs/blob/main/1BM24CS137_Week6_Propositional_Logic/1BM24CS137_Week6_Propositional_Logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import re
import itertools

def evaluate_clause(clause, model, statement_map=None):
    """Evaluates a propositional logic clause/sentence containing standard logic symbols by translating them to Python."""
    expr = clause

    if statement_map:
        for key in sorted(statement_map.keys(), key=lambda x: len(x), reverse=True):
            if key in expr:
                expr = expr.replace(key, f"({statement_map[key]})")

    expr = expr.replace('<=>', ' == ')

    while '=>' in expr:
        parts = expr.rsplit('=>', 1)
        expr = f"(not ({parts[0]})) or ({parts[1]})"

    expr = re.sub(r'~\s*([a-zA-Z0-9_]+|\()', r' not \1', expr)
    expr = expr.replace('&', ' and ')
    expr = expr.replace('|', ' or ')

    try:
        return bool(eval(expr, {}, model))
    except Exception as e:
        raise ValueError(f"Error evaluating expression '{expr}' derived from '{clause}': {e}")

def pl_true(sentence, model, statement_map=None):
    """Return True if the sentence is true in the model."""
    return evaluate_clause(sentence, model, statement_map)

def generate_truth_table(kb_statements, alpha, symbols, statement_map=None):
    """Generates and prints the complete truth table, verifying the entailment relationship."""
    headers = list(symbols) + [f"KB_S{i+1}" for i in range(len(kb_statements))] + ["KB_Total", "Query (alpha)", "Valid Entailment?"]
    col_widths = [max(len(h), 6) for h in headers]

    header_line = " | ".join(f"{h:^{col_widths[i]}}" for i, h in enumerate(headers))
    print("\n" + header_line)
    print("-" * len(header_line))

    entails = True

    for combination in itertools.product([True, False], repeat=len(symbols)):
        model = dict(zip(symbols, combination))

        kb_vals = [pl_true(clause, model, statement_map) for clause in kb_statements]
        kb_total = all(kb_vals)

        query_val = pl_true(alpha, model, statement_map)

        row_entails_check = True
        if kb_total and not query_val:
            row_entails_check = False
            entails = False

        row_data = (
            [str(model[s])[0] for s in symbols] +
            [str(v)[0] for v in kb_vals] +
            [str(kb_total)[0], str(query_val)[0], "Yes" if row_entails_check else "NO (VIOLATION)"]
        )

        row_line = " | ".join(f"{val:^{col_widths[i]}}" for i, val in enumerate(row_data))
        print(row_line)

    return entails

def tt_entails(kb_statements, alpha, symbols, statement_map=None):
    """Implementation of the TT-ENTAILS? algorithm with explicit truth table visualization."""
    return generate_truth_table(kb_statements, alpha, symbols, statement_map)

def main():
    print("--- TT-ENTAILS Logic Checker (with Truth Table Visualization) ---")
    print("Operators supported: ~ (NOT), & (AND), | (OR), => (IMPLIES), <=> (BICONDITIONAL), ⊨ or |= (ENTAILS)")
    print("Example KB statement: (A | C) & (B | ~C)\n")

    vars_input = input("Enter all propositional variables separated by spaces (e.g. A B C): ")
    symbols = [v.strip() for v in vars_input.split() if v.strip()]

    num_statements = int(input("Enter the number of statements in KB (e.g. 2): "))

    kb_statements = []
    statement_map = {}
    print("\nEnter your statements:")
    for i in range(num_statements):
        label = f"S{i+1}"
        statement = input(f"Enter statement {label}: ")
        kb_statements.append(statement)
        statement_map[label] = statement

    print("\nYour labeled statements:")
    for label, stmt in statement_map.items():
        print(f"{label}: {stmt}")

    num_queries = int(input("\nEnter the number of queries (entailments) to evaluate: "))

    for q in range(num_queries):
        print(f"\n--- Evaluating Query {q+1} ---")
        print("You can check standard queries (e.g., S2) or entailment relations (e.g., S1⊨S2 or S1|=S2).")
        alpha_input = input(f"Enter query {q+1}: ").strip()

        entail_symbols = ["⊨", "|="]
        used_symbol = None
        for sym in entail_symbols:
            if sym in alpha_input:
                used_symbol = sym
                break

        if used_symbol:
            premise, conclusion = alpha_input.split(used_symbol, 1)
            premise = premise.strip()
            conclusion = conclusion.strip()
            print(f"Checking if '{premise}' entails '{conclusion}'...")
            result = tt_entails([premise], conclusion, symbols, statement_map)
            print(f"\nResult: {premise} {used_symbol} {conclusion} is {result}")
        else:
            print("Running TT-ENTAILS algorithm against KB...")
            result = tt_entails(kb_statements, alpha_input, symbols, statement_map)
            print(f"\nDoes the KB entail '{alpha_input}'? Result: {result}")

if __name__ == '__main__':
    main()

--- TT-ENTAILS Logic Checker (with Truth Table Visualization) ---
Operators supported: ~ (NOT), & (AND), | (OR), => (IMPLIES), <=> (BICONDITIONAL), ⊨ or |= (ENTAILS)
Example KB statement: (A | C) & (B | ~C)

Enter all propositional variables separated by spaces (e.g. A B C): A B C
Enter the number of statements in KB (e.g. 2): 2

Enter your statements:
Enter statement S1: (A|C)&(B|~C)
Enter statement S2: A|B

Your labeled statements:
S1: (A|C)&(B|~C)
S2: A|B

Enter the number of queries (entailments) to evaluate: 1

--- Evaluating Query 1 ---
You can check standard queries (e.g., S2) or entailment relations (e.g., S1⊨S2 or S1|=S2).
Enter query 1: (S1, S2)
Running TT-ENTAILS algorithm against KB...

  A    |   B    |   C    | KB_S1  | KB_S2  | KB_Total | Query (alpha) | Valid Entailment?
-----------------------------------------------------------------------------------------
  T    |   T    |   T    |   T    |   T    |    T     |       T       |        Yes       
  T    |   T    |   F 